In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/prksh830/suicide-dataset/generalized_suicide_dataset_with_domain.csv


In [3]:
!pip install transformers
!pip install sentencepiece
!pip install shap
!pip install lime
!pip install nrclex
!pip install textblob

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from transformers import (
    AutoTokenizer,
    AutoModel
)

from sklearn.model_selection import (
    train_test_split
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from textblob import TextBlob

from nrclex import NRCLex

import matplotlib.pyplot as plt

In [5]:
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [6]:
MODEL_NAME = "vinai/bertweet-base"

MAX_LEN = 128

BATCH_SIZE = 16

EPOCHS = 10

LEARNING_RATE = 2e-5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(DEVICE)

cuda


In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
!find /kaggle/input -name "*.csv"

/kaggle/input/datasets/prksh830/suicide-dataset/generalized_suicide_dataset_with_domain.csv


In [9]:
df = pd.read_csv(
    "/kaggle/input/datasets/prksh830/suicide-dataset/generalized_suicide_dataset_with_domain.csv"
)

print(df.head())

print(df.shape)

                                          clean_text  label  token_count  \
0  always made me feel worthless but all of a sud...      1           14   
1                      hope i die in my sleep tonite      1            7   
2         my life sucks just want to give up and cry      1           10   
3  breed of queensland male who should ve already...      1           49   
4  closer to killing myself everyday famsquad but...      1           14   

     domain  
0  dataset2  
1  dataset2  
2  dataset2  
3  dataset2  
4  dataset2  
(57306, 4)


In [ ]:
def extract_emotion_features(text):

    emotion = NRCLex(text)

    scores = emotion.affect_frequencies

    sadness = scores.get("sadness", 0)

    fear = scores.get("fear", 0)

    anger = scores.get("anger", 0)

    disgust = scores.get("disgust", 0)

    joy = scores.get("joy", 0)

    trust = scores.get("trust", 0)

    return [
        sadness,
        fear,
        anger,
        disgust,
        joy,
        trust
    ]

In [14]:
from nrclex import NRCLex

In [15]:
def extract_emotion_features(text):

    emotion = NRCLex(text)

    scores = emotion.raw_emotion_scores

    emotions = [
        "sadness",
        "fear",
        "anger",
        "disgust",
        "joy",
        "trust"
    ]

    feature_vector = []

    for emo in emotions:

        if emo in scores:
            feature_vector.append(scores[emo])

        else:
            feature_vector.append(0)

    return feature_vector

In [17]:
sample = NRCLex("I feel hopeless")

print(dir(sample))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lexicon__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_build_word_affect', '_compute_top_emotions', '_resolve_lexicon', 'load_raw_text', 'load_token_list']


In [18]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [19]:
def extract_emotion_features(text):

    try:

        result = emotion_classifier(text[:512])[0]

        emotion_scores = {
            item['label']: item['score']
            for item in result
        }

        sadness = emotion_scores.get("sadness", 0)

        fear = emotion_scores.get("fear", 0)

        anger = emotion_scores.get("anger", 0)

        disgust = emotion_scores.get("disgust", 0)

        joy = emotion_scores.get("joy", 0)

        surprise = emotion_scores.get("surprise", 0)

        return [
            sadness,
            fear,
            anger,
            disgust,
            joy,
            surprise
        ]

    except:

        return [0, 0, 0, 0, 0, 0]

In [20]:
emotion_features = df["clean_text"].apply(
    extract_emotion_features
)

emotion_features = np.array(
    emotion_features.tolist()
)

print(emotion_features.shape)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


(57306, 6)


In [25]:
texts_list = df["clean_text"].astype(str).tolist()

results = emotion_classifier(
    texts_list[:],
    batch_size=32,
    truncation=True
)

In [ ]:
emotion_features = []

for result in results:

    emotion_scores = {
        item['label']: item['score']
        for item in result
    }i

    sadness = emotion_scores.get("sadness", 0)

    fear = emotion_scores.get("fear", 0)

    anger = emotion_scores.get("anger", 0)

    disgust = emotion_scores.get("disgust", 0)

    joy = emotion_scores.get("joy", 0)

    surprise = emotion_scores.get("surprise", 0)

    emotion_features.append([
        sadness,
        fear,
        anger,
        disgust,
        joy,
        surprise
    ])

emotion_features = np.array(
    emotion_features
)

print(emotion_features.shape)

In [ ]:
def extract_behavioral_features(text):

    words = text.split()

    sentence_length = len(words)

    exclamation_count = text.count("!")

    question_count = text.count("?")

    first_person_count = (
        text.lower().count("i ")
        + text.lower().count("me ")
        + text.lower().count("my ")
    )

    negative_words = [
        "die",
        "death",
        "suicide",
        "kill",
        "hopeless",
        "worthless",
        "depressed",
        "pain"
    ]

    negative_count = sum(
        word in text.lower()
        for word in negative_words
    )

    return [
        sentence_length,
        exclamation_count,
        question_count,
        first_person_count,
        negative_count
    ]

In [ ]:
behavioral_features = df["clean_text"].astype(str).apply(
    extract_behavioral_features
)

behavioral_features = np.array(
    behavioral_features.tolist()
)

print(behavioral_features.shape)

In [ ]:
texts = df["clean_text"].astype(str).values

labels = df["label"].values

print(texts.shape)

print(labels.shape)

In [ ]:
from sklearn.model_selection import train_test_split

(
    train_texts,
    test_texts,
    train_labels,
    test_labels,
    train_emotion,
    test_emotion,
    train_sentiment,
    test_sentiment,
    train_behavior,
    test_behavior
) = train_test_split(

    texts,
    labels,
    emotion_features,
    sentiment_features,
    behavioral_features,

    test_size=0.2,

    stratify=labels,

    random_state=42
)

In [ ]:
class MUSEDataset(Dataset):

    def __init__(
        self,
        texts,
        emotion,
        sentiment,
        behavior,
        labels
    ):

        self.texts = texts

        self.emotion = emotion

        self.sentiment = sentiment

        self.behavior = behavior

        self.labels = labels

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = tokenizer(

            text,

            truncation=True,

            padding='max_length',

            max_length=MAX_LEN,

            return_tensors='pt'
        )

        return {

            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "emotion":
                torch.tensor(
                    self.emotion[idx],
                    dtype=torch.float
                ),

            "sentiment":
                torch.tensor(
                    self.sentiment[idx],
                    dtype=torch.float
                ),

            "behavior":
                torch.tensor(
                    self.behavior[idx],
                    dtype=torch.float
                ),

            "label":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [ ]:
train_dataset = MUSEDataset(

    train_texts,

    train_emotion,

    train_sentiment,

    train_behavior,

    train_labels
)

test_dataset = MUSEDataset(

    test_texts,

    test_emotion,

    test_sentiment,

    test_behavior,

    test_labels
)

In [ ]:
class MUSENet(nn.Module):

    def __init__(self):

        super(MUSENet, self).__init__()

        # ==================================
        # BERTWEET SEMANTIC ENCODER
        # ==================================

        self.bert = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # ==================================
        # SEMANTIC PROJECTION
        # ==================================

        self.semantic_fc = nn.Linear(
            768,
            256
        )

        # ==================================
        # MULTIMODAL FUSION
        # ==================================

        self.fusion_fc1 = nn.Linear(
            256 + 6 + 2 + 5,
            128
        )

        self.dropout = nn.Dropout(0.3)

        self.fusion_fc2 = nn.Linear(
            128,
            64
        )

        # ==================================
        # ATTENTION LAYER
        # ==================================

        self.attention = nn.Linear(
            64,
            64
        )

        # ==================================
        # CLASSIFIER
        # ==================================

        self.classifier = nn.Linear(
            64,
            2
        )

    def forward(

        self,

        input_ids,

        attention_mask,

        emotion,

        sentiment,

        behavior
    ):

        # ==============================
        # BERTWEET EMBEDDINGS
        # ==============================

        outputs = self.bert(

            input_ids=input_ids,

            attention_mask=attention_mask
        )

        cls_embedding = outputs.last_hidden_state[:,0,:]

        semantic = self.semantic_fc(
            cls_embedding
        )

        # ==============================
        # FEATURE CONCATENATION
        # ==============================

        fused = torch.cat(

            [
                semantic,
                emotion,
                sentiment,
                behavior
            ],

            dim=1
        )

        # ==============================
        # FUSION NETWORK
        # ==============================

        x = F.relu(
            self.fusion_fc1(fused)
        )

        x = self.dropout(x)

        x = F.relu(
            self.fusion_fc2(x)
        )

        # ==============================
        # ATTENTION
        # ==============================

        attention_weights = torch.softmax(

            self.attention(x),

            dim=1
        )

        x = x * attention_weights

        # ==============================
        # CLASSIFICATION
        # ==============================

        output = self.classifier(x)

        return output

In [ ]:
class FocalLoss(nn.Module):

    def __init__(
        self,
        alpha=0.75,
        gamma=2
    ):

        super(FocalLoss, self).__init__()

        self.alpha = alpha

        self.gamma = gamma

    def forward(
        self,
        inputs,
        targets
    ):

        ce_loss = F.cross_entropy(

            inputs,

            targets,

            reduction='none'
        )

        pt = torch.exp(-ce_loss)

        focal_loss = (

            self.alpha *

            (1 - pt) ** self.gamma *

            ce_loss
        )

        return focal_loss.mean()

In [ ]:
model = MUSENet().to(DEVICE)

criterion = FocalLoss()

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE
)

In [ ]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        emotion = batch["emotion"].to(DEVICE)

        sentiment = batch["sentiment"].to(DEVICE)

        behavior = batch["behavior"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(

            input_ids,

            attention_mask,

            emotion,

            sentiment,

            behavior
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Loss: {avg_loss:.4f}"
    )

In [ ]:
model.eval()

predictions = []

probabilities = []

actuals = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch[
            "input_ids"
        ].to(DEVICE)

        attention_mask = batch[
            "attention_mask"
        ].to(DEVICE)

        emotion = batch["emotion"].to(DEVICE)

        sentiment = batch["sentiment"].to(DEVICE)

        behavior = batch["behavior"].to(DEVICE)

        labels = batch["label"].to(DEVICE)

        outputs = model(

            input_ids,

            attention_mask,

            emotion,

            sentiment,

            behavior
        )

        probs = torch.softmax(
            outputs,
            dim=1
        )

        preds = torch.argmax(
            probs,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        probabilities.extend(
            probs[:,1].cpu().numpy()
        )

        actuals.extend(
            labels.cpu().numpy()
        )

In [ ]:
accuracy = accuracy_score(
    actuals,
    predictions
)

precision = precision_score(
    actuals,
    predictions
)

recall = recall_score(
    actuals,
    predictions
)

f1 = f1_score(
    actuals,
    predictions
)

auc = roc_auc_score(
    actuals,
    probabilities
)

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1-Score :", f1)

print("AUROC    :", auc)

In [ ]:
cm = confusion_matrix(
    actuals,
    predictions
)

plt.figure(figsize=(6,6))

plt.imshow(cm)

plt.title("Confusion Matrix")

plt.colorbar()

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
torch.save(

    model.state_dict(),

    "MUSE_Net_Model.pt"
)

print("Model Saved Successfully")